# Simple Vector Indexing Demo (Before vs After)

This notebook shows indexing in the simplest possible way:

1. Store a few vectors in a vector database (ChromaDB).
2. Search **without an index** (check every vector).
3. Search **with a simple index** (check only one group).

Goal: understand the idea, not production performance.

## Inputs: example documents

We’ll start with a tiny set of text snippets. Each snippet will become a vector, and then we’ll search for the most similar snippets to a query.

In [ ]:
import chromadb
import numpy as np

# Tiny dataset for teaching
docs = [
    "book flight ticket",
    "cancel booking refund",
    "hotel room reservation",
    "baggage allowance policy",
    "airport security rules",
    "meal options onboard",
]
ids = [f"d{i}" for i in range(len(docs))]

## Convert text to vectors (teaching embedding)

To keep things simple, we use a deterministic embedding function.
It turns each text into 3 numbers:
- travel score
- policy score
- service score

No external model download is needed.

In [ ]:
# Very small deterministic embedding: 3 keyword buckets
vocab = {
    "travel": {"flight", "ticket", "airport", "booking", "reservation"},
    "policy": {"policy", "rules", "allowance", "security"},
    "service": {"meal", "onboard", "refund", "cancel", "hotel"},
}

def embed(text: str):
    # Keep embeddings tiny + deterministic for teaching (no ML model download).
    words = text.lower().split()

    # Each output dimension is a count of how many words match a bucket.
    # Order: [travel, policy, service]
    vec = [
        sum(w in vocab[k] for w in words)
        for k in ["travel", "policy", "service"]
    ]
    return np.array(vec, dtype=float)

embeddings = [embed(d).tolist() for d in docs]

## Store vectors in ChromaDB

We create an in-memory Chroma collection and insert our document vectors, then we load the stored `documents` and `embeddings` back from Chroma.

That way, the later “before vs after” search logic is working over the vectors as they live in the database.

In [ ]:
client = chromadb.Client()
collection = client.get_or_create_collection(name="simple_index_demo")

# Store vectors in ChromaDB.
collection.upsert(ids=ids, documents=docs, embeddings=embeddings)
print("Stored", collection.count(), "vectors in ChromaDB")

# Load them back so the “before vs after indexing” code is using the DB data.
data = collection.get(include=["documents", "embeddings"])
docs = data["documents"]
embeddings = data["embeddings"]

## Before indexing: linear scan

This checks the query against **every vector**.

In [ ]:
def linear_search(query, top_k=2):
    q = embed(query)
    scores = []
    checks = 0

    # Linear scan = compute distance to every stored vector.
    for i, e in enumerate(embeddings):
        checks += 1
        dist = np.linalg.norm(q - np.array(e))  # Euclidean distance
        scores.append((dist, docs[i]))

    scores.sort(key=lambda x: x[0])
    return scores[:top_k], checks

query = "flight rules"
linear_top, linear_checks = linear_search(query)
print("Query:", query)
print("Top results:", linear_top)
print("Distance checks:", linear_checks)

## After indexing: toy grouped index (conceptual)

A **toy index** is a tiny shortcut we build so we don't compare a query against **every** stored vector.

### Analogy: “open only one folder”
Think of each document as a small card. We create **3 folders** (Travel / Policy / Service).

- **Before indexing (linear scan):** for every question, you open **all** folders and compare against **all** cards.
- **After indexing (toy grouped index):** you look at the question and guess the most likely folder, then you only compare against cards inside that folder.

That is what `index_groups` is doing:

- We pre-partition vectors into 3 buckets using a simple rule (the strongest embedding dimension).
- For a new query, we predict which bucket it belongs to and compute distances **only inside that bucket**.

### Trade-off (why it’s a “toy”)
If the guess is wrong, we do much less work—but we may miss the true best match.

This is not a real ANN index like IVF/HNSW, but it captures the core idea: **indexing prunes the search space**, so distance computations drop.

## Compare before vs after indexing

Now we’ll compare how many distance computations each method performed:
- linear scan (no index)
- grouped index search (toy index)

In [ ]:
index_groups = {0: [], 1: [], 2: []}

# Toy index = pre-partition vectors into 3 buckets.
# We decide the bucket using the embedding dimension with the largest value.
for i, e in enumerate(embeddings):
    group = int(np.argmax(e))  # 0=travel-ish, 1=policy-ish, 2=service-ish
    index_groups[group].append(i)

def indexed_search(query, top_k=2):
    q = embed(query)
    target_group = int(np.argmax(q))

    # Instead of comparing with every vector, compare only inside this group.
    candidate_ids = index_groups[target_group]

    scores = []
    checks = 0
    for i in candidate_ids:
        checks += 1
        # Compute distance only inside the selected candidate bucket.
        dist = np.linalg.norm(q - np.array(embeddings[i]))
        scores.append((dist, docs[i]))

    scores.sort(key=lambda x: x[0])
    return scores[:top_k], checks, target_group

indexed_top, indexed_checks, used_group = indexed_search(query)
print("Used group:", used_group)
print("Top results:", indexed_top)
print("Distance checks:", indexed_checks)

In [ ]:
# These numbers help students see the benefit of pruning (indexing).
print("\nComparison")
print("- Linear scan checks:", linear_checks)
print("- Indexed search checks:", indexed_checks)
print("- Saved checks:", linear_checks - indexed_checks)

## Takeaway

- **Without index**: easy, but checks every vector.
- **With index**: extra structure, but fewer checks.
- Real vector databases use stronger index types (like IVF, HNSW), but the core idea is the same.